In [33]:
import os
import pygmt
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import colors
import numpy as np
import pandas as pd 
import glob 

%load_ext autoreload 
%autoreload 2
%matplotlib inline
import utils
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [34]:
cell_dict = {
    0.01: (20, 4096),
    0.008: (20, 4096),
    0.0063: (20, 4096),
    0.005: (10, 8192),
    0.0039: (10, 8192),
    0.003: (5, 16384),
    0.002: (5, 16384),
    0.0014: (2.5, 32768),
    0.001: (2.5, 32768)    
} # DX and N2 info for each mdoel 

In [35]:
#### this is the only cell that needs changing before running
main_dir = "model_outputs/single_fault_benchmarks/inplane/"
files_interest = ["output_Dc0039/"]
dc = 0.0039  
spin_up = 200
loading_type = "In plane"

DX, N2 = cell_dict[dc]
DX = [DX]
N2 = [N2]
cols = [
    "Index",
    "Displacement",
    "Col3",
    "Col4",
    "Col5",
    "col",
    "cplb",
    "Slip Rate",
    "Col7",
    "Col8",
    "Col9",
    "Col10",
    "Col11",
]
# time step conversion
time_step_rate = 20  # --export-netcdf-rate 20 \ ----- I am using the same in all models so this one shouldn't change..
grid_step_rate_horizontal = 4

feature = "log10v"  # tau, slip, log10v
fault1_path = main_dir + "fault-01-" + feature + ".grd"

In [36]:
time_step_rate = 20
grid_step_rate_horizontal = 4

for i, (file, DXi, N2i) in enumerate(zip(files_interest, DX, N2)):
    file = file

    # load time file for times
    file_pattern = main_dir + file + "patch-01-*.dat"
    matching_files = glob.glob(file_pattern)
    fault_data = pd.read_csv(matching_files[0], sep="\s+", header=None, names=cols)

    # load grid for moment
    feature = "log10v"  # tau, slip, log10v
    fault1_path = main_dir + file + "fault-01-" + feature + ".grd"
    grid_fault1 = pygmt.load_dataarray(fault1_path, engine="netcdf4")
    delta_grid = time_step_rate
    time_grid = fault_data["Index"].iloc[:: int(delta_grid)].values

    # make catalog
    grid1_masked = np.ma.masked_where(
        grid_fault1 < -3.5, grid_fault1
    )  # subset areas where velocity>seismic slip to isolate events
    grid1_masked = grid1_masked.filled(
        np.nan
    )  # for viz in seaborn fill unmasked areas with nan
    (
        timestep_event_fault1,
        rupture_length_pixels_fault1,
        group_id_fault1,
        total_pixels_fault1_xdir,
        pixel_size,
    ) = utils.find_events(grid1_masked, DXi, N2i, "No")
    time_f1 = time_grid[timestep_event_fault1]
    rupture_length_fault1 = np.array(rupture_length_pixels_fault1)

    # remove spin-up period
    time_cutoff = spin_up * 365 * 24 * 60 * 60
    cutoff_index = np.where(time_grid >= time_cutoff)[0][0]
    time_grid = time_grid[cutoff_index:]
    time_f1 = time_f1[time_f1 >= time_cutoff]
    rupture_length_fault1 = rupture_length_fault1[
        len(rupture_length_fault1) - len(time_f1) :
    ]

    # remove ruptures that are 4 cells only (1 cell in here because of MTC downsampling) -- artifacts from masking
    ok_size_rupture_idx1 = np.where(rupture_length_fault1 > 3)[0]
    time_f1 = time_f1[ok_size_rupture_idx1]
    rupture_length_fault1 = rupture_length_fault1[ok_size_rupture_idx1]

    # remove partial ruptures
    time_f1 = time_f1[rupture_length_fault1 * pixel_size > 4500]

    # measure interevent times
    inter_event_times = utils.measure_interevent_time_f(time_f1)  # in seconds
    inter_event_times_years = inter_event_times / (365 * 24 * 60 * 60)  # in years

    times_file = "code_output_data/interevent_times_benchmarks.csv"
    if not os.path.isfile(times_file):
        cols = [
            "Loading",
            "Dc",
            "Inter-event times (seconds)",
            "Inter-event times (years)",
        ]
        times_database = pd.DataFrame(columns=cols)
        times_database.to_csv(times_file, index=False)
    else:
        times_database = pd.read_csv(times_file)
    existing_combination = times_database[
        (times_database["Dc"] == dc) & (times_database["Loading"] == loading_type)
    ]
    if not existing_combination.empty:
        times_database = times_database[
            ~(
                (times_database["Dc"] == dc)
                & (times_database["Loading"] == loading_type)
            )
        ]
        print(f"Removed existing rows with Dc={dc}.")

    times_sim_i = pd.DataFrame(
        {
            "Loading": [loading_type],
            "Dc": [dc],
            "Inter-event times (seconds)": [", ".join(map(str, inter_event_times))],
            "Inter-event times (years)": [", ".join(map(str, inter_event_times_years))],
        }
    )

    new_rows = pd.concat([times_sim_i], ignore_index=True)
    catalog_df = pd.concat([times_database, new_rows], ignore_index=True)
    catalog_df.to_csv(times_file, index=False)